In [ ]:
import numpy as np
import pandas as pd
import torch, random, math, json
import torch.nn as nn
#import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torch.nn         import functional as F
from torch            import LongTensor, Tensor, nn
from sklearn          import preprocessing
from pandas.tseries.offsets import MonthEnd
from einops import rearrange, repeat, einsum
from typing           import Union, Optional, Tuple, Any, Iterable, NamedTuple, TypeAlias, cast
from dataclasses      import dataclass


In [ ]:
# 2. utility fun.
def create_sequences(X, d_time, sld_len):
    tot_len = X.shape[0]
    nos     = (tot_len - d_time) // sld_len
    idx_start = np.asarray(range(nos)) * sld_len
    i  = 0; X2 = np.zeros((nos, d_time, X.shape[1]))    # tot_len / sld_len
    for idx in idx_start:
        X2[i,:,:] = X[idx:idx+d_time,:]
        i = i+1
    return np.array(X2, dtype=np.float32)

# loss function
def calc_wmae(predict, targets, now, nom, noq, wew, weq, wet, itv) -> torch.Tensor:
    masks   = (~torch.isnan(targets)).type(torch.float32)
    targets = torch.nan_to_num(targets, nan=0)
    mf_wgt  = torch.cat(( torch.ones(1,now)*wew, torch.ones(1,nom), torch.ones(1,noq)*weq ), 1)
    mf_wgt[0,itv] = mf_wgt[0,itv]*wet
    w_mask   = masks*mf_wgt
    return torch.sum(torch.abs(predict -targets)*w_mask) / (torch.sum(masks) +1e-12)

# mixed freq. module
class MixfWQLayer4(nn.Module):
    """ Weekly - Monthly - Quarterly layer
    12 weeks / quarter, 4 wow dataset
    """
    def __init__(self, n: int, now: int, nom: int):
        super().__init__()
        # Create the weight matrix using torch methods
        noq   = n-now-nom
        mm_wq = torch.cat([
            torch.cat([torch.eye(now), torch.zeros((20*n -now, now))], dim=0),
            torch.cat([0*torch.eye(now, nom), 1/4*torch.eye(n, nom), 1/4*torch.eye(n, nom),
                       1/4*torch.eye(n, nom), 1/4*torch.eye(n-now, nom), 0*torch.eye(20*n-4*n, nom)], dim=0),
            torch.cat([0*torch.eye(now+nom, noq), 1/12*torch.eye(n, noq),  1/12*torch.eye(n, noq),  1/12*torch.eye(n, noq),  1/12*torch.eye(n, noq),
                         2/12*torch.eye(n, noq),  2/12*torch.eye(n, noq),  2/12*torch.eye(n, noq),  2/12*torch.eye(n, noq),
                         3/12*torch.eye(n, noq),  3/12*torch.eye(n, noq),  3/12*torch.eye(n, noq),  3/12*torch.eye(n, noq),
                         2/12*torch.eye(n, noq),  2/12*torch.eye(n, noq),  2/12*torch.eye(n, noq),  2/12*torch.eye(n, noq),
                         1/12*torch.eye(n, noq),  1/12*torch.eye(n, noq),  1/12*torch.eye(n, noq),  1/12*torch.eye(n-now-nom, noq),], dim=0),], dim=1)
        self.w = nn.Parameter(mm_wq, requires_grad=False)             # Weight is not trainable
        self.b = nn.Parameter(torch.zeros(n), requires_grad=False)    # Bias is not trainable

    def forward(self, inputs):
        # inputs : batch x length x features
        t = inputs.shape[1]
        n = inputs.shape[2]
        inputs_and_lags = []
        for i in range(0, 20):  # i = 0, ..., 19
            mask = torch.cat([torch.zeros(i, n), torch.ones(t-i, n)], dim=0)
            rolled_inputs = torch.mul(torch.roll(inputs, shifts=i, dims=1), mask)
            inputs_and_lags.append(rolled_inputs)
        inputs_and_lags = torch.cat(inputs_and_lags, dim=2)

        return torch.matmul(inputs_and_lags, self.w) + self.b

# make a pseudo realtime data
def load_rts(date_rel, wago, idx_trgt, nor, d_step):
    nov      = date_rel.shape[1]
    idx_miss = np.zeros((nor, nov), dtype=bool)  # Initialize with False values
    norh = nor-36

    if nor < 36:
        for i in range(nor):
            for j in range(nov):
                idx_miss[i, j] = date_rel.iloc[0,idx_trgt] - (date_rel.iloc[0,j] + d_step * (i - (nor-1))) - wago * 7 < 0
    else:
        for i in range(norh,nor):
            for j in range(nov):
                idx_miss[i, j] = date_rel.iloc[0,idx_trgt] - (date_rel.iloc[0,j] + d_step * (i - (nor-1))) - wago * 7 < 0
    return idx_miss


In [ ]:
# wkly: load excel data
dir_root = "/Users/kym08/My Drive/code/python/moef_2025/"
data_all = pd.read_csv(f"{dir_root}data.w.y.csv", parse_dates=['datew'])       # ~ x 46
#data_all = pd.read_csv(f"{dir_root}data.w.i.csv", parse_dates=['datew'])       # ~ x 43

data_r   = data_all.set_index(['datem', 'datew'], append=True)
data_w   = data_all.iloc[0:,1:].set_index("datew")
#data_w2  = data_all.iloc[0:,1:-1]

# match the release date to loaded data
spec_r   = pd.read_csv(f"{dir_root}spec.w.y.csv")     # !!! not necessary to change
print(data_r.shape)

rel_day1 = data_w.iloc[1:1,:].T
rel_day2 = spec_r.set_index('SeriesID')['Release']
release  = rel_day1.join(rel_day2).T

#  3. data_loader
opm     = 4             # obs / month
d_time  = opm*12*3      # each dataset length, 3 year
sld_len = opm*3         # skipping length,     3 month

def create_sequences(X, d_time, sld_len):
    tot_len = X.shape[0]
    nos     = (tot_len - d_time) // sld_len
    idx_start = np.asarray(range(nos)) * sld_len
    i  = 0; X2 = np.zeros((nos, d_time, X.shape[1]))    # tot_len / sld_len
    for idx in idx_start:
        X2[i,:,:] = X[idx:idx+d_time,:]
        i = i+1
    return np.array(X2, dtype=np.float32)

raw_train = data_r[pd.to_datetime(data_r.index.get_level_values(1)) <= pd.Timestamp("2020-01-01")]
scaler = preprocessing.StandardScaler().fit(raw_train)                          # numpy
x_r    = scaler.transform(raw_train)
X_r    = create_sequences(x_r, d_time, sld_len)
split  = int(len(X_r) * 0.8)
X_train, X_test = X_r[:split], X_r[split:]

train_loader= DataLoader(TensorDataset(torch.from_numpy(X_train)), batch_size=32, shuffle = True, num_workers=0)    # torch.Dataloader
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test)), shuffle = False, num_workers=0)    # torch.Dataloader


#### define model

In [ ]:
# 4. Mamba
'''
Glossary:
    b          : batch size            (`B` in Mamba paper [1] Algorithm 2)
    l          : sequence length       (`L` in [1] Algorithm 2)
    d / d_model: hidden dim
    n / d_state: latent state dim      (`N` in [1] Algorithm 2)
    expand     : expansion factor      (`E` in [1] Section 3.4)
    d_in / d_inner: d*expand           (`D` in [1] Algorithm 2)
    A, B, C, D : state space parameters(See any state space representation formula)
                                       (B, C are input-dependent (aka selective, a key innovation in Mamba); A, D are not)
    Δ / delta  : input-dependent step size
    dt_rank    : rank of Δ             (See [1] Section 3.6 "Parameterization of ∆")
'''
from __future__ import annotations

@dataclass
class ModelArgs:
    d_model  : int
    n_layer  : int
    d_output : int       # modified, renamed from vocab_size
    now      : int       # added; of wkly vars
    nom      : int       # added; of mkly vars
    noq      : int       # added; of qtly vars
    mf_layer : str       
    d_state  : int = 16
    expand   : int = 2
    dt_rank  : Union[int, str] = 'auto'
    d_conv   : int = 4 
    #pad_vocab_size_multiple: int = 8 # modified
    conv_bias: bool = True
    bias     : bool = False
    
    def __post_init__(self):
        self.d_inner = int(self.expand * self.d_model)
        if self.dt_rank == 'auto':
            self.dt_rank = math.ceil(self.d_model / 16)
    #    if self.d_output % self.pad_vocab_size_multiple != 0:
    #        self.d_output += (self.pad_vocab_size_multiple - self.d_output % self.pad_vocab_size_multiple)

class ResidualBlock(nn.Module):
    def __init__(self, args: ModelArgs):
        """Simple block wrapping Mamba block with normalization and residual connection."""
        super().__init__()
        self.args = args
        self.mixer = MambaBlock(args)
        self.norm = RMSNorm(args.d_model)
    def forward(self, x):
        """
        Args: x: shape (b, l, d)    (See Glossary at top for definitions of b, l, d_in, n...)
        Returns: output: shape (b, l, d)
        Official Implementation:
            Block.forward(), https://github.com/state-spaces/mamba/blob/main/mamba_ssm/modules/mamba_simple.py#L297
            Note: the official repo chains residual blocks that look like
                [Add -> Norm -> Mamba] -> [Add -> Norm -> Mamba] -> [Add -> Norm -> Mamba] -> ...
            where the first Add is a no-op. This is purely for performance reasons as this
            allows them to fuse the Add->Norm.
            We instead implement our blocks as the more familiar, simpler, and numerically equivalent
                [Norm -> Mamba -> Add] -> [Norm -> Mamba -> Add] -> [Norm -> Mamba -> Add] -> ....
        """
        #output = self.mixer(self.norm(x)) + x
        output0, C = self.mixer(self.norm(x))
        output = output0 + x
        return output, C

class MambaBlock(nn.Module):
    def __init__(self, args: ModelArgs):
        """A single Mamba block, as described in Figure 3 in Section 3.4 in the Mamba paper [1]."""
        super().__init__()
        self.args = args
        self.in_proj= nn.Linear(args.d_model, args.d_inner*2, bias=args.bias)
        self.conv1d = nn.Conv1d(in_channels=args.d_inner, out_channels=args.d_inner, bias=args.conv_bias,
                                kernel_size=args.d_conv, groups=args.d_inner, padding=args.d_conv - 1,)
        # x_proj takes in `x` and outputs the input-specific Δ, B, C
        self.x_proj = nn.Linear(args.d_inner, args.dt_rank + args.d_state * 2, bias=False)
        # dt_proj projects Δ from dt_rank to d_in
        self.dt_proj = nn.Linear(args.dt_rank, args.d_inner, bias=True)
        A = repeat(torch.arange(1, args.d_state + 1), 'n -> d n', d=args.d_inner)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(args.d_inner))
        self.out_proj = nn.Linear(args.d_inner, args.d_model, bias=args.bias)

    def forward(self, x):
        """Mamba block forward. This looks the same as Figure 3 in Section 3.4 in the Mamba paper [1].
        Args: x: shape (b, l, d)    (See Glossary at top for definitions of b, l, d_in, n...)
        Returns: output: shape (b, l, d)
        Official Implementation:
            class Mamba, https://github.com/state-spaces/mamba/blob/main/mamba_ssm/modules/mamba_simple.py#L119
            mamba_inner_ref(), https://github.com/state-spaces/mamba/blob/main/mamba_ssm/ops/selective_scan_interface.py#L311
        """
        (b, l, d) = x.shape
        x_and_res = self.in_proj(x)  # shape (b, l, 2 * d_in)
        (x, res) = x_and_res.split(split_size=[self.args.d_inner, self.args.d_inner], dim=-1)
        x = rearrange(x, 'b l d_in -> b d_in l')
        x = self.conv1d(x)[:, :, :l]
        x = rearrange(x, 'b d_in l -> b l d_in')
        x = F.silu(x)
        #y = self.ssm(x)
        y, C  = self.ssm(x)
        #cache.C[self.layer_idx].copy_(C) 
        y = y * F.silu(res)
        output = self.out_proj(y)
        return output, C
    
    def ssm(self, x):
        """Runs the SSM. See:
            - Algorithm 2 in Section 3.2 in the Mamba paper [1]
            - run_SSM(A, B, C, u) in The Annotated S4 [2]
        Args: x: shape (b, l, d_in)    (See Glossary at top for definitions of b, l, d_in, n...)
        Returns: output: shape (b, l, d_in)
        Official Implementation:
            mamba_inner_ref(), https://github.com/state-spaces/mamba/blob/main/mamba_ssm/ops/selective_scan_interface.py#L311
        """
        (d_in, n) = self.A_log.shape
        # Compute ∆ A B C D, the state space parameters.
        #     A, D are input independent (see Mamba paper [1] Section 3.5.2 "Interpretation of A" for why A isn't selective)
        #     ∆, B, C are input-dependent (this is a key difference between Mamba and the linear time invariant S4,
        #                                  and is why Mamba is called **selective** state spaces)
        A = -torch.exp(self.A_log.float())  # shape (d_in, n)
        D = self.D.float()
        x_dbl = self.x_proj(x)  # (b, l, dt_rank + 2*n)
        (delta, B, C) = x_dbl.split(split_size=[self.args.dt_rank, n, n], dim=-1)  # delta: (b, l, dt_rank). B, C: (b, l, n)
        delta = F.softplus(self.dt_proj(delta))  # (b, l, d_in)
        y = self.selective_scan(x, delta, A, B, C, D)  # This is similar to run_SSM(A, B, C, u) in The Annotated S4 [2]
        return y, C
    
    def selective_scan(self, u, delta, A, B, C, D):
        """ selective scan algorithm. See:
            - Section 2 State Space Models in the Mamba paper [1]
            - Algorithm 2 in Section 3.2 in the Mamba paper [1]
            - run_SSM(A, B, C, u) in The Annotated S4 [2]

        This is the classic discrete state space formula:
            x(t + 1) = Ax(t) + Bu(t)
            y(t)     = Cx(t) + Du(t)
        except B and C (and the step size delta, which is used for discretization) are dependent on the input x(t).
        Args:
            u: shape (b, l, d_in)    (See Glossary at top for definitions of b, l, d_in, n...)
            delta: shape (b, l, d_in)
            A: shape (d_in, n)
            B: shape (b, l, n)
            C: shape (b, l, n)
            D: shape (d_in,)
        Returns:
            output: shape (b, l, d_in)
        Official Implementation:
            selective_scan_ref(), https://github.com/state-spaces/mamba/blob/main/mamba_ssm/ops/selective_scan_interface.py#L86
            Note: I refactored some parts out of `selective_scan_ref` out, so the functionality doesn't match exactly.
        """
        (b, l, d_in) = u.shape
        n = A.shape[1]
        # Discretize continuous parameters (A, B)
        # - A is discretized using zero-order hold (ZOH) discretization (see Section 2 Equation 4 in the Mamba paper [1])
        # - B is discretized using a simplified Euler discretization instead of ZOH. From a discussion with authors:
        #   "A is the more important term and the performance doesn't change much with the simplification on B"
        deltaA   = torch.exp(einsum(delta, A, 'b l d_in, d_in n -> b l d_in n'))
        deltaB_u = einsum(delta, B, u, 'b l d_in, b l n, b l d_in -> b l d_in n')
        
        # Perform selective scan (see scan_SSM() in The Annotated S4 [2])
        # Note that the below is sequential, while the official implementation does a much faster parallel scan that
        # is additionally hardware-aware (like FlashAttention).
        x = torch.zeros((b, d_in, n), device=deltaA.device) # x means h
        ys = []    
        for i in range(l):
            x = deltaA[:, i] * x + deltaB_u[:, i]
            y = einsum(x, C[:, i, :], 'b d_in n, b n -> b d_in')
            ys.append(y)
        y = torch.stack(ys, dim=1)  # shape (b, l, d_in)
        y = y + u * D
        return y

class RMSNorm(nn.Module):
# root mean-square norm : normalize only volatility
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
    def forward(self, x):
        output = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight
        return output

class Mamba(nn.Module):
    def __init__(self, args: ModelArgs):
        """Full Mamba model."""
        super().__init__()
        self.args = args
        #self.embedding= nn.Embedding(args.d_output, args.d_model)
        self.embedding= nn.Linear(args.d_output*2, args.d_model)
        self.layers   = nn.ModuleList([ResidualBlock(args) for _ in range(args.n_layer)])
        self.norm_f   = RMSNorm(args.d_model)
        self.lm_head  = nn.Linear(args.d_model, args.d_output, bias=False) # False
        #self.lm_head.weight = self.embedding.weight  # modified; Tie output projection to embedding weights. See "Weight Tying" paper

        self.mq_layer = MixfWQLayer4(args.d_output, args.d_output-args.noq-args.nom, args.nom) 
    
    def forward(self, x):
        """
        Args: input_ids (long tensor): shape (b, l)    (See Glossary at top for definitions of b, l, d_in, n...)
        Returns:logits: shape (b, l, vocab_size)
        Official Implementation:
            class MambaLMHeadModel, https://github.com/state-spaces/mamba/blob/main/mamba_ssm/models/mixer_seq_simple.py#L173

        """
        mask   = (~torch.isnan(x)).type(torch.float32) # added
        xr     = torch.nan_to_num(x, nan=0)            # added
        x0     = torch.cat([xr, mask], dim=2)          # added; batch x d_time x nov

        x = self.embedding(x0)
        for layer in self.layers:
            x, f = layer(x)
        #x = self.norm_f(x)                  # modified !!
        logits0 = self.lm_head(x)

        x_hat  = self.mq_layer(logits0)      # added
        logits = (mask*xr + (1-mask)*x_hat)  # added

        return logits, x_hat, f
    
    @staticmethod
    def from_pretrained(pretrained_model_name: str):
        """Load pretrained weights from HuggingFace into model.
        Args: pretrained_model_name: One of
                * 'state-spaces/mamba-2.8b-slimpj'
        Returns: model: Mamba model with weights loaded
        """
        from transformers.utils import WEIGHTS_NAME, CONFIG_NAME
        from transformers.utils.hub import cached_file
        
        def load_config_hf(model_name):
            resolved_archive_file = cached_file(model_name, CONFIG_NAME, _raise_exceptions_for_missing_entries=False)
            return json.load(open(resolved_archive_file))
        
        def load_state_dict_hf(model_name, device=None, dtype=None):
            resolved_archive_file = cached_file(model_name, WEIGHTS_NAME, _raise_exceptions_for_missing_entries=False)
            return torch.load(resolved_archive_file, weights_only=True, map_location='cpu', mmap=True)
        
        config_data = load_config_hf(pretrained_model_name)
        args = ModelArgs(d_model=config_data['d_model'], n_layer=config_data['n_layer'], d_output=config_data['vocab_size'])
        model = Mamba(args)
        state_dict = load_state_dict_hf(pretrained_model_name)
        new_state_dict = {}
        for key in state_dict:
            new_key = key.replace('backbone.', '')
            new_state_dict[new_key] = state_dict[key]
        model.load_state_dict(new_state_dict)
        return model

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#args     = ModelArgs(d_output=110, n_layer=2, d_model=37, d_state = 7, now = 0, noq = 8, dt_rank = 3, mf_layer='mq')
args     = ModelArgs(d_output=45, n_layer=2, d_model=16, d_state = 5, now = 7, nom = 38, noq = 0, dt_rank = 2, mf_layer='wmq') # output
#args     = ModelArgs(d_output=43, n_layer=2, d_model=15, d_state = 5, now = 8, noq = 0, dt_rank = 2, mf_layer='wmq') # investment
model    = Mamba(args)
loss_fn  = calc_wmae #nn.MSELoss()



#### Training

In [ ]:
# 5. SWA training
SAVE_PATH = 'mamba.w.y.pth'
torch.manual_seed(1); #random.seed(1); torch.cuda.manual_seed_all(1)
#iv  = release.columns.get_loc('KOIPALL.G')  #
epochs    = 200
#now=0; nom=102; noq=0; wew=0.225; weq=1; wet=1; itv=110-8
now=7; nom=38; noq=0; wew=0.225; weq=1; wet=1; itv=26-1  #output
#now=8; nom=35; noq=0; wew=0.225; weq=1; wet=1; itv=41-1  # investment

optimizer = torch.optim.SGD(model.parameters(), lr=0.2, momentum=0.9) # lr = 1e-3, mmt = 0
swa_start = int(0.5*epochs)                    # start SWA at 50% of training
swa_model = AveragedModel(model).to(device)    # running average of weights
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=swa_start)
swa_scheduler = SWALR(optimizer, swa_lr=0.2, anneal_epochs=10, anneal_strategy='cos')  # steady small LR during SWA phase, swa_lr=0.05

for epoch in range(epochs):
    # 1. Training Phase
    model.train()
    train_loss = 0.0
    in_swa_phase = (epoch >= swa_start)
    for ib, xb in enumerate(train_loader):
        xb = xb[0]                          # list to torch
        optimizer.zero_grad()
        impt, pred, f = model(xb)
        loss = loss_fn(pred, xb, now, nom, noq, wew, weq, wet, itv)
        loss.backward()
        optimizer.step()
        if in_swa_phase:
            swa_model.update_parameters(model)
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_loader.dataset)  # type: ignore

    # 2. SWA phase: accumulate averaged parameters + step SWA LR
    if in_swa_phase: swa_scheduler.step()
    else:                scheduler.step()

    # 3. Testing (Validation) Phase
    eval_model = swa_model if in_swa_phase else model
    eval_model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for ib, xb in enumerate(test_loader):
            xb = xb[0]
            impt, pred, f = eval_model(xb)     # Before swa_start, evaluate the standard model.
            loss = loss_fn(pred, xb, now, nom, noq, wew, weq, wet, itv)
            test_loss += loss.item() * len(xb)
    test_loss /= len(test_loader.dataset)

    # 4. Logging
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.3f}, Test Loss: {test_loss:.3f}")

# Recompute BatchNorm for the SWA-averaged weights
update_bn(train_loader, swa_model)
swa_model.eval() # Use the averaged weights for saving/eval

print("--- Final SWA Model Evaluation ---")
final_test_loss = 0
with torch.no_grad():
    for ib, xb in enumerate(test_loader):
        xb = xb[0]
        #xb = xb[0] if isinstance(batch, (list, tuple)) else xb; xb = xb.to(device,non_blocking=True)
        impt, pred, f = swa_model(xb) # Use the final, updated SWA model
        loss = loss_fn(pred, xb, now, nom, noq, wew, weq, wet, itv)
        final_test_loss += loss.item() * len(xb)
final_test_loss /= len(test_loader.dataset)

torch.save({'model_state_dict'    : model.state_dict(),
            'swa_model_state_dict': swa_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            }, SAVE_PATH)

print(f"Epoch {epoch+1}: SWA model saved.\n"
      f"Final train loss (last epoch): {train_loss:.3f}\n"
      f"✅Final SWA Test Loss: {final_test_loss:.3f}")


# multiple quarters with pseudo real-time dataset
m_name='mamba'
tvr  = 'KOIPALL.G'                        # KOGDP...D  KOCNPER.D  KOGFCF..D  /   KOIPALL.G

frq = 1                                  # 3 qtly var, 1 mthly var
dps = 31 if opm == 1 else 7              # 31 for monthly model, 7 weekly model
iv  = release.columns.get_loc(tvr)    
ny  = 1                                   # of years to test, 9 or 4
nm  = 9                                  # months to test after years
btw = 18 if frq == 3 else 8
idx_ib  = pd.Series(range(btw, -1, -1))    # 18 16 ... 1 (0) weeks ago

X4  = np.zeros((btw+1, int(3/frq)*4*ny +int(nm/frq)))                  #  noy years + 2 quarters, (gdp 4, pce 12) x noy

m_dates   = pd.to_datetime(data_r.index.get_level_values(1).drop_duplicates()) if opm == 4 else [] #dropna()
oos_dates = list(data_r.index[data_r.index >= pd.Timestamp("2020-01-01")]) if opm == 1 else list(m_dates[m_dates>=pd.Timestamp('2024-01-01')]) # !! 2020-01-01

for cc, vv in enumerate(pd.Series(range(0, frq*int(3/frq)*4*ny +frq*int(nm/frq), frq))):    # (gdp 4, pce 12) x noy
    date_now = oos_dates[3-int(3/frq) +frq*cc]                            # (gdp 2, pce 0) + (gdp 3, pce 1) x cc
    X3 = np.zeros((btw+1, d_time, data_r.shape[1]))   # B x L x N : 19 rt step as Batch, d_time = L, 
    i  = 0; 
    for ib in idx_ib:                                                             # 18~0 weeks ago to given quarter
        data_n  = data_r[data_r.index<=pd.Timestamp(date_now)].iloc[-d_time:,:] if opm == 1 else data_r[pd.to_datetime(data_r.index.get_level_values(1))<=date_now].iloc[-d_time:,:]   # dataset for given quarter cut to last seq_len
        idx_m   = load_rts(release, ib, iv, d_time, dps)       # monthly 31, weekly 7 ; masking for missing as True
        data_n[idx_m==True] = np.nan
        Xn      = scaler.transform(data_n) # hide for RevIn
        #Xn      = scaler.transform(data_n) +scaler.mean_/scaler.scale_
        #Xn      = data_n.to_numpy()
        X3[i,:,:] = Xn                                                            #
        i = i+1
    #datasetn   = {"X": X3};             
    imputation, x_hat, x_state = swa_model(torch.from_numpy(X3).type(torch.float32))  # impute the originally-missing values and real-time missing values
    imput_dn = (imputation.detach().numpy()*scaler.scale_)+scaler.mean_
    #imput_dn = imputation # RevIn
    X4[:,cc] = imput_dn[:,-1,iv]                                                # each quarter cc to columns

rst0 = pd.DataFrame(X4); rst0.to_csv(f"{dir_root}/result/nowcast.w.y."+tvr+"_"+m_name+".csv")

err  = X4[-1,:] - X4[-2,:];  errx = np.delete(err,[1,2],0)     # exclude '20.1Q  / [0,1,2]
rmse = np.sqrt(np.mean(np.square(err)));  mae  = np.mean(np.abs(err));  print('RMSE1:',round(rmse,2), 'MAE1:',round(mae,2))
rmse = np.sqrt(np.mean(np.square(errx))); mae  = np.mean(np.abs(errx)); print('RMSEx:',round(rmse,2), 'MAEx:',round(mae,2))
